In [1]:
import pandas as pd
import numpy as np

#load data 
image_df = pd.read_csv('data/images.csv')

style_df = pd.read_csv('data/styles_FIXED.csv')
style_df = style_df.drop(columns=['Unnamed: 10','Unnamed: 11'])



In [2]:
# importing some more packages 
import requests
import matplotlib.pyplot as plt 

In [3]:
# get only tops, bottoms, and shoes from style df
allowed_vals = ['Topwear', 'Bottomwear', 'Shoes']
filtered_style = style_df[style_df["subCategory"].isin(allowed_vals)]

# separate styles into topwear, bottomwear, and shoes dfs
shoes_style = filtered_style[filtered_style["subCategory"] == "Shoes"]
topWear_style = filtered_style[filtered_style["subCategory"] == "Topwear"]
bottomWear_style = filtered_style[filtered_style["subCategory"] == "Bottomwear"]

In [4]:
# get random samples of each 
# n_samples = 30 
n_samples = 500
samp_btm = bottomWear_style.sample(n=n_samples, random_state=42)
samp_top = topWear_style.sample(n=n_samples, random_state=42)
samp_shoes = shoes_style.sample(n=n_samples, random_state=42)


---


TEST DATA 

In [34]:
# get test samples for each category and make sure not in training sample
n_samples = 6
test_btm = bottomWear_style[~bottomWear_style.index.isin(samp_btm.index)].sample(n=n_samples, random_state=42)
test_top = topWear_style[~topWear_style.index.isin(samp_top.index)].sample(n=n_samples, random_state=42)
test_shoes = shoes_style[~shoes_style.index.isin(samp_shoes.index)].sample(n=n_samples, random_state=42)

# check that test not in sample
assert not test_btm.isin(samp_btm).any().any()
assert not test_top.isin(samp_top).any().any()
assert not test_shoes.isin(samp_shoes).any().any()

#concat test samples into one df
test_df = pd.concat([test_btm, test_top, test_shoes], ignore_index=True)


# get rows from image df for each category
test_images = image_df[image_df["filename"].isin(test_df["id"].to_list())]

assert len(test_images) == n_samples * 3

# merge test images with test df to get category labels
test_merged = pd.merge(test_images, test_df, left_on='filename', right_on='id')

In [36]:
# df to csv
test_merged.to_csv('data/test_data.csv', index=False)

In [ ]:
# subcatogories and labels for test
import pandas as pd

df = pd.read_csv('data/test_data.csv')
# combine adult and child genders
df.loc[df['gender'].isin(['Girls','Women']),'gender'] = 'Female'
df.loc[df['gender'].isin(['Boys','Men']),'gender'] = 'Male'


# going to create a new label for the CNN; the label will consist of gender, season, and subcategory!
pd.set_option('display.max_rows', None)
df['img_label'] = df['subCategory'] + ' ' + df['season'] + ' ' + df['gender']
df['img_label'].value_counts()

# save the new df with labels to csv
df.to_csv('data/labeled_test_data.csv', index=False)

----

In [26]:
# remove extensions from filename column 
image_df["filename"]  = image_df["filename"].str.replace(".jpg","")
image_df["filename"]  = pd.to_numeric(image_df["filename"])


In [6]:
#get rows to match with sampled bottomwear from img df
vals = samp_btm["id"].to_list()
samp_btm_imgs = image_df[image_df["filename"].isin(vals)]

#get rows to match with sampled topwear from img df
vals = samp_top["id"].to_list()
samp_top_imgs = image_df[image_df["filename"].isin(vals)]

#get rows to match with sampled shoes from img df
vals = samp_shoes["id"].to_list()
samp_shoes_imgs = image_df[image_df["filename"].isin(vals)]


# concat images to one df
final_imgs_df = pd.concat([samp_btm_imgs, samp_top_imgs, samp_shoes_imgs], ignore_index=True)

# concat styles to one df
final_styles_df = pd.concat([samp_btm, samp_top, samp_shoes], ignore_index=True)

In [7]:
# merge images df with styles df
final_df = pd.merge(final_imgs_df, final_styles_df,left_on = "filename", right_on = "id")
final_df.head()

,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black
1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts
2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans
3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings
4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants


In [8]:
final_df.shape

(1500, 12)

In [9]:
from PIL import Image
from io import BytesIO
def load_image(url):
    try:
        response = requests.get(url, timeout=10)
        return Image.open(BytesIO(response.content)).convert("RGB")
    except:
        return None

final_df["image_data"] = final_df["link"].apply(load_image)

In [10]:
final_df

,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_data
0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black,<PIL.Image.Image image mode=RGB size=1800x2400...
1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts,<PIL.Image.Image image mode=RGB size=1080x1440...
2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans,<PIL.Image.Image image mode=RGB size=1800x2400...
3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings,<PIL.Image.Image image mode=RGB size=1800x2400...
4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants,<PIL.Image.Image image mode=RGB size=1080x1440...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,17667,http://assets.myntassets.com/v1/images/style/p...,17667,Women,Footwear,Shoes,Casual Shoes,Black,Winter,2015.0,Casual,Catwalk Women Buckel Black Casual Shoe,<PIL.Image.Image image mode=RGB size=1800x2400...
1496,5687,http://assets.myntassets.com/v1/images/style/p...,5687,Men,Footwear,Shoes,Casual Shoes,Brown,Winter,2015.0,Casual,Skechers Men's Traffics Brown Shoe,<PIL.Image.Image image mode=RGB size=1080x1440...
1497,41822,http://assets.myntassets.com/v1/images/style/p...,41822,Women,Footwear,Shoes,Sports Shoes,White,Summer,2012.0,Sports,Skechers Women Grey & White Sports Shoes,<PIL.Image.Image image mode=RGB size=1080x1440...
1498,17694,http://assets.myntassets.com/v1/images/style/p...,17694,Men,Footwear,Shoes,Casual Shoes,Navy Blue,Fall,2011.0,Casual,Vans Men Authentic Navy Blue Casual Shoes,<PIL.Image.Image image mode=RGB size=1800x2400...


In [11]:
final_df.to_csv('/Users/aniyahmcwilliams/Spring 2026/final_df.csv')

In [ ]:
# checking the shape
final_df.shape


(1500, 13)

In [ ]:
# ! pip install tensorflow

  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 MB 5.2 MB/s  0:00:44m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 4.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 4.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 4.6 MB/s  0:00:00
Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl (429 kB)
Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 5.3 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [tensorflow]0 [tensorflow]


In [26]:
# importing the necessary packages for the model 
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
# for the first model going to attempt to do a resnet18 model 
from torchvision.models import resnet18, ResNet18_Weights
from torch.nn import Module

In [27]:
# need to see the number of classes in articletype 
final_df['articleType'].value_counts()

articleType
Tshirts               221
Casual Shoes          194
Sports Shoes          135
Jeans                 116
Trousers              105
Shirts                101
Shorts                 92
Heels                  84
Kurtas                 63
Tops                   58
Track Pants            52
Flats                  44
Formal Shoes           43
Leggings               31
Capris                 31
Skirts                 26
Jackets                12
Sweatshirts            10
Sweaters               10
Stockings               9
Kurtis                  8
Churidar                7
Patiala                 7
Jeggings                7
Tunics                  5
Salwar and Dupatta      4
Salwar                  4
Dupatta                 4
Suspenders              3
Tracksuits              3
Tights                  3
Swimwear                3
Blazers                 2
Belts                   1
Rompers                 1
Rain Jacket             1
Name: count, dtype: int64

In [28]:
# need to get the number of classes in the in the articletype clothing 
num_of_classes = final_df['articleType'].nunique()
print(num_of_classes)

36


In [30]:
# setting up the parameters for the ResNet structutre
batch_size = 32
epochs = 200
data_augmentation = True
subtract_pixel_mean = True

class MyCNN1(Module):
    def __init__(self):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT # going to use the default resnet weights 
        self.resnet18 = resnet18(weights=weights, progress=False).eval()
        self.transforms = weights.transforms(antialias=True)

    def forward(self, X):
        with torch.no_grad():
            X = self.transforms(X)
            y_pred = self.resnet18(X)
            return y_pred.argmax(dim=1)

In [31]:
model1 = MyCNN1().to('cpu') # only doing cpu for now 
print(model1) # okay yay the model is good 

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/aniyahmcwilliams/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
MyCNN1(
  (resnet18): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2

_______

In [ ]:
import pandas as pd
import numpy as np 
import requests
from PIL import Image
from io import BytesIO
import tensorflow as tf


df = pd.read_csv('data/final_df.csv')

In [ ]:
# train and test with stratify for subcategories 

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['subCategory'])

In [ ]:
# turn all the images into numpy arrays and put them into a list
arr = []
fail = []

for link in train_df['link']:
    try: 
        response = requests.get(link, stream=True).raw
        img = Image.open(response)
        tensor = np.array(img).astype('float32')
        tensor = np.expand_dims(tensor, axis=0)
        arr.append(tensor)
    except:
        fail.append(link)
        continue
    
